# FAC_NB_Preparar_Control — catálogo y bitácora de la ingesta metadata-driven

Este cuaderno **prepara el terreno** de la demo de ingesta dirigida por metadatos. Todo es **sintético y didáctico**: no usa datos reales ni conexiones externas.

## Qué hace (y es idempotente: se puede ejecutar todas las veces que haga falta)

1. Crea el esquema `fac` (solo si el Lakehouse tiene esquemas habilitados) y dos tablas Delta de control:
   - `fac_catalogo_ingesta`: **qué** se carga y **cómo** (una fila por tabla destino, con su marca de agua).
   - `fac_bitacora_ingesta`: **qué pasó** en cada corrida (una fila por tabla y corrida).
2. Copia el CSV de transacciones que dejó la Sesión 1 a una carpeta propia de la demo (`Files/fac_demo/entrada/transacciones/`). Se copia, **no se modifica el original**: las guías de la Sesión 2 verifican su hash.
3. Siembra el catálogo con MERGE: si ya existe una fila **no pisa** su `ultima_marca` ni su `activo`.
4. Según `accion`, puede además **generar un archivo de incremento** (`incremento`) o **reiniciar la demo** (`reiniciar`).

## Parámetros (celda siguiente)

| Parámetro | Valores | Para qué |
|---|---|---|
| `accion` | `preparar` / `incremento` / `reiniciar` | Qué hacer además de la preparación base |
| `destino` | `auto` / `ninguno` / nombre de esquema | `auto` usa el esquema `fac` si el Lakehouse tiene esquemas; si no, tablas sin esquema |
| `ruta_base` | ruta bajo `Files` | Donde la Sesión 1 dejó los CSV (`Files/bronze/banca_sintetica`) |
| `ruta_demo` | ruta bajo `Files` (debe empezar por `Files/fac_`) | Carpeta propia de la demo |
| `numero_incremento` | 1 a 20 | Cada número genera un archivo distinto, con fechas posteriores |

**Requisitos:** el Lakehouse del laboratorio (por ejemplo `AB_Lakehouse`) adjunto como **predeterminado** y los CSV de la Sesión 1 cargados en `ruta_base`.

In [ ]:
# Celda de parámetros: un pipeline o la API pueden sobrescribir estos valores.
accion = "preparar"            # preparar | incremento | reiniciar
destino = "auto"               # auto | ninguno | nombre de esquema
ruta_base = "Files/bronze/banca_sintetica"
ruta_demo = "Files/fac_demo"
numero_incremento = 1

## Paso 1 — Contexto y validaciones

Se valida cada parámetro **antes** de tocar nada, y se confirma que hay un Lakehouse predeterminado adjunto. Los nombres que se usarán dentro de sentencias SQL solo pueden ser identificadores simples.

In [ ]:
import json
import re
import uuid
import datetime
from pyspark.sql import functions as F

PATRON_IDENT = re.compile(r"[A-Za-z][A-Za-z0-9_]{0,63}\Z")


def validar_ident(valor, campo):
    """Los nombres se interpolan en SQL: solo se aceptan identificadores simples."""
    if not isinstance(valor, str) or PATRON_IDENT.fullmatch(valor) is None:
        raise ValueError(
            f"{campo}: se esperaba un identificador (letra inicial; letras, números y _); se recibió {valor!r}"
        )
    return valor


def resolver_esquema(destino):
    """auto = 'fac' si el Lakehouse tiene esquemas habilitados, si no ninguno; 'ninguno' o un nombre explícito."""
    d = (destino or "auto").strip()
    if d.lower() == "ninguno":
        return ""
    if d.lower() == "auto":
        try:
            nombres = {str(r[0]).lower() for r in spark.sql("SHOW SCHEMAS").collect()}
        except Exception:
            nombres = set()
        return "fac" if "dbo" in nombres else ""
    return validar_ident(d, "destino")


def calificar(esquema, tabla):
    return f"{esquema}.{tabla}" if esquema else tabla

import csv
import io
import random
import notebookutils

accion = (accion or "").strip().lower()
if accion not in ("preparar", "incremento", "reiniciar"):
    raise ValueError(f"accion debe ser preparar, incremento o reiniciar; se recibió {accion!r}")

ruta_base = (ruta_base or "").strip().rstrip("/")
ruta_demo = (ruta_demo or "").strip().rstrip("/")
if not ruta_base.startswith("Files/"):
    raise ValueError("ruta_base debe ser una ruta relativa que empiece por 'Files/'")
if not ruta_demo.startswith("Files/fac_") or ".." in ruta_demo:
    raise ValueError("ruta_demo debe empezar por 'Files/fac_' (así la demo nunca toca otras carpetas)")

numero_incremento = int(numero_incremento)
if not 1 <= numero_incremento <= 20:
    raise ValueError("numero_incremento debe estar entre 1 y 20")

ctx = notebookutils.runtime.context
if not ctx.get("defaultLakehouseId"):
    raise RuntimeError("Adjunta el Lakehouse del laboratorio como predeterminado y reinicia la sesión.")
print("Lakehouse predeterminado:", ctx.get("defaultLakehouseName"))

ESQUEMA = resolver_esquema(destino)
if ESQUEMA:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ESQUEMA}")
CAT = calificar(ESQUEMA, "fac_catalogo_ingesta")
BIT = calificar(ESQUEMA, "fac_bitacora_ingesta")
print("Esquema destino:", ESQUEMA or "(ninguno)", "| catálogo:", CAT, "| bitácora:", BIT)

## Paso 2 — Tablas de control (DDL idempotente)

`CREATE TABLE IF NOT EXISTS` no falla ni modifica nada si las tablas ya existen.

**Catálogo** (`fac_catalogo_ingesta`): `id`, `tabla_destino`, `ruta_origen`, `formato`, `tipo_carga` (`completa` | `incremental`), `columna_marca`, `ultima_marca`, `activo` (1/0), `orden` y, como columna adicional respecto al mínimo, `columna_clave` (la llave de negocio que permite que el incremental sea **idempotente**).

> Los textos vacíos se guardan como `''` (no como NULL) a propósito: así el pipeline nunca recibe un valor nulo al leer el catálogo.

**Bitácora** (`fac_bitacora_ingesta`): `run_id`, `tabla`, `inicio`, `fin`, `filas_leidas`, `filas_escritas`, `marca_antes`, `marca_despues`, `estado`, `mensaje`.

In [ ]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CAT} (
    id INT,
    tabla_destino STRING,
    ruta_origen STRING,
    formato STRING,
    tipo_carga STRING,
    columna_marca STRING,
    columna_clave STRING,
    ultima_marca STRING,
    activo INT,
    orden INT
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BIT} (
    run_id STRING,
    tabla STRING,
    inicio TIMESTAMP,
    fin TIMESTAMP,
    filas_leidas BIGINT,
    filas_escritas BIGINT,
    marca_antes STRING,
    marca_despues STRING,
    estado STRING,
    mensaje STRING
) USING DELTA
""")
print("Tablas de control listas.")

## Paso 3 — Datos de la demo: el catálogo "semilla"

Cuatro tablas activas y una inactiva (para mostrar cómo `activo = 0` la excluye de la corrida):

| Tabla destino | Origen | Tipo | Marca / llave |
|---|---|---|---|
| `fac_bronze_sucursales`, `fac_bronze_productos`, `fac_bronze_clientes` | CSV de la Sesión 1 (maestras) | `completa` (se reemplaza todo) | — |
| `fac_bronze_transacciones` | carpeta de entrada de la demo | `incremental` | `fecha_hora` / `transaccion_id` |
| `fac_bronze_calendario` | CSV de la Sesión 1 | `completa`, **inactiva** | — |

Las transacciones apuntan a una **carpeta**, no a un archivo: así un archivo nuevo que "llega" se lee junto con el anterior y el filtro por marca de agua se queda solo con lo nuevo.

In [ ]:
ENTRADA_TX = f"{ruta_demo}/entrada/transacciones"
BASE_TX = f"{ENTRADA_TX}/transacciones_base.csv"

SEMILLA = [
    dict(id=1, tabla_destino="fac_bronze_sucursales", ruta_origen=f"{ruta_base}/sucursales.csv", formato="csv",
         tipo_carga="completa", columna_marca="", columna_clave="", activo=1, orden=10),
    dict(id=2, tabla_destino="fac_bronze_productos", ruta_origen=f"{ruta_base}/productos.csv", formato="csv",
         tipo_carga="completa", columna_marca="", columna_clave="", activo=1, orden=20),
    dict(id=3, tabla_destino="fac_bronze_clientes", ruta_origen=f"{ruta_base}/clientes.csv", formato="csv",
         tipo_carga="completa", columna_marca="", columna_clave="", activo=1, orden=30),
    dict(id=4, tabla_destino="fac_bronze_transacciones", ruta_origen=f"{ENTRADA_TX}/", formato="csv",
         tipo_carga="incremental", columna_marca="fecha_hora", columna_clave="transaccion_id", activo=1, orden=40),
    dict(id=5, tabla_destino="fac_bronze_calendario", ruta_origen=f"{ruta_base}/calendario.csv", formato="csv",
         tipo_carga="completa", columna_marca="", columna_clave="", activo=0, orden=50),
]
for fila in SEMILLA:
    validar_ident(fila["tabla_destino"], "tabla_destino")
print(f"{len(SEMILLA)} filas semilla definidas.")

## Paso 4 — Reiniciar la demo (solo si `accion = "reiniciar"`)

Deja la demo como al principio: borra **únicamente** las tablas `fac_bronze_*` del catálogo, vacía catálogo y bitácora y elimina los archivos `transacciones_incremento_*.csv` de la carpeta de entrada. Los CSV de la Sesión 1 y el archivo base copiado **no se tocan**. El Paso 6 vuelve a sembrar el catálogo.

In [ ]:
if accion == "reiniciar":
    nombres = {fila["tabla_destino"] for fila in SEMILLA}
    nombres |= {r[0] for r in spark.table(CAT).select("tabla_destino").collect()}
    for t in sorted(nombres):
        if not t.startswith("fac_bronze_"):
            print("Omitida (no es fac_bronze_*):", t)
            continue
        validar_ident(t, "tabla_destino")
        spark.sql(f"DROP TABLE IF EXISTS {calificar(ESQUEMA, t)}")
        print("Tabla eliminada:", calificar(ESQUEMA, t))
    spark.sql(f"DELETE FROM {CAT}")
    spark.sql(f"DELETE FROM {BIT}")
    if notebookutils.fs.exists(ENTRADA_TX):
        for f in notebookutils.fs.ls(ENTRADA_TX):
            if f.name.startswith("transacciones_incremento_"):
                notebookutils.fs.rm(f.path, False)
                print("Incremento eliminado:", f.name)
    print("Demo reiniciada: catálogo y bitácora vacíos.")
else:
    print("Paso omitido (accion =", accion + ")")

## Paso 5 — Carpeta de entrada de transacciones

Se **copia** `transacciones.csv` (lo que dejó la Sesión 1) a `Files/fac_demo/entrada/transacciones/transacciones_base.csv`. Si la copia ya existe no se vuelve a copiar. Los maestros se leen directamente de la ruta de la Sesión 1.

Antes de seguir se comprueba que existan los CSV que necesitan las tablas **activas**.

In [ ]:
notebookutils.fs.mkdirs(ENTRADA_TX)
if not notebookutils.fs.exists(BASE_TX):
    origen_tx = f"{ruta_base}/transacciones.csv"
    if not notebookutils.fs.exists(origen_tx):
        raise FileNotFoundError(f"No existe {origen_tx}. Completa los accionables de la Sesión 1 (CSV en {ruta_base}).")
    notebookutils.fs.cp(origen_tx, BASE_TX)
    print("Copiado:", origen_tx, "->", BASE_TX)
else:
    print("Ya existe:", BASE_TX)

faltantes = []
for fila in SEMILLA:
    if fila["activo"] == 1:
        ruta = fila["ruta_origen"].rstrip("/")
        ok = notebookutils.fs.exists(ruta)
        print("OK " if ok else "FALTA", ruta)
        if not ok:
            faltantes.append(ruta)
if faltantes:
    raise FileNotFoundError(f"Faltan orígenes de tablas activas: {faltantes}")

## Paso 6 — Sembrar el catálogo con MERGE

- Fila nueva → se inserta con `ultima_marca = ''` (carga inicial).
- Fila existente → solo se actualizan los datos de configuración (ruta, formato, tipo, columnas, orden). **No** se tocan `id`, `ultima_marca` ni `activo`, de modo que volver a ejecutar este cuaderno no "olvida" la marca de agua ni reactiva una tabla que el facilitador desactivó.

In [ ]:
SCHEMA_CAT = ("id INT, tabla_destino STRING, ruta_origen STRING, formato STRING, tipo_carga STRING, "
              "columna_marca STRING, columna_clave STRING, ultima_marca STRING, activo INT, orden INT")
filas = [(f["id"], f["tabla_destino"], f["ruta_origen"], f["formato"], f["tipo_carga"], f["columna_marca"],
          f["columna_clave"], "", f["activo"], f["orden"]) for f in SEMILLA]
spark.createDataFrame(filas, SCHEMA_CAT).createOrReplaceTempView("_fac_semilla")

spark.sql(f"""
MERGE INTO {CAT} AS c
USING _fac_semilla AS s
ON c.tabla_destino = s.tabla_destino
WHEN MATCHED THEN UPDATE SET
    c.ruta_origen = s.ruta_origen,
    c.formato = s.formato,
    c.tipo_carga = s.tipo_carga,
    c.columna_marca = s.columna_marca,
    c.columna_clave = s.columna_clave,
    c.orden = s.orden
WHEN NOT MATCHED THEN INSERT *
""")
print("Catálogo sembrado.")

## Paso 7 — Generar un archivo de incremento (solo si `accion = "incremento"`)

Crea `transacciones_incremento_NN.csv` en la carpeta de entrada: **48 transacciones sintéticas** (3 días × 16) con fechas **posteriores** a las de la carga inicial (octubre de 2026). Es determinista: el mismo `numero_incremento` produce siempre el mismo archivo.

- Mismo encabezado y formato que `transacciones.csv`; solo valores limpios (sin duplicados ni errores deliberados).
- Identificadores `TX000800xx` (+1000 por cada número de incremento): no chocan con los 72.072 identificadores del dataset original.
- Cuentas dentro del rango existente (`CTA000001` a `CTA001800`), tipos y direcciones según el diccionario de datos.
- El incremento 1 cubre del 1 al 3 de octubre de 2026, el 2 del 4 al 6, y así sucesivamente.

Volver a generar el mismo número **sobrescribe** el mismo archivo con el mismo contenido (idempotente).

In [ ]:
ENCABEZADO = ["transaccion_id", "cuenta_id", "fecha_hora", "tipo_transaccion", "direccion", "canal",
              "monto", "moneda", "estado"]
TIPOS_TX = [("SALARIO", "CREDITO"), ("DEPOSITO", "CREDITO"), ("TRANSFERENCIA_RECIBIDA", "CREDITO"),
            ("INTERES", "CREDITO"), ("RETIRO", "DEBITO"), ("PAGO_SERVICIO", "DEBITO"), ("COMPRA", "DEBITO"),
            ("TRANSFERENCIA_ENVIADA", "DEBITO"), ("COMISION", "DEBITO")]
CANALES = ["BANCA_MOVIL", "BANCA_MOVIL", "BANCA_MOVIL", "WEB", "ATM", "SUCURSAL"]


def generar_incremento(n):
    rng = random.Random(20261112 + n)
    primer_dia = datetime.date(2026, 10, 1) + datetime.timedelta(days=3 * (n - 1))
    filas = []
    for dia in range(3):
        fecha = primer_dia + datetime.timedelta(days=dia)
        for _ in range(16):
            hora = datetime.time(rng.randint(6, 21), rng.randint(0, 59), rng.randint(0, 59))
            tipo, direccion = rng.choice(TIPOS_TX)
            centimos = rng.randint(50_000, 15_000_000)
            filas.append((datetime.datetime.combine(fecha, hora), rng.randint(1, 1800), tipo, direccion,
                          rng.choice(CANALES), centimos))
    filas.sort(key=lambda f: f[0])
    salida = io.StringIO()
    escritor = csv.writer(salida, lineterminator="\n")
    escritor.writerow(ENCABEZADO)
    for k, (ts, cuenta, tipo, direccion, canal, centimos) in enumerate(filas, start=1):
        escritor.writerow([f"TX{80000 + (n - 1) * 1000 + k:08d}", f"CTA{cuenta:06d}", ts.isoformat(timespec="seconds"),
                           tipo, direccion, canal, f"{centimos // 100}.{centimos % 100:02d}", "CRC", "APLICADA"])
    return salida.getvalue(), len(filas), filas[0][0], filas[-1][0]


if accion == "incremento":
    texto, cantidad, primera, ultima = generar_incremento(numero_incremento)
    destino_inc = f"{ENTRADA_TX}/transacciones_incremento_{numero_incremento:02d}.csv"
    notebookutils.fs.put(destino_inc, texto, True)
    print(f"Incremento generado: {destino_inc}")
    print(f"  filas: {cantidad} | primera fecha_hora: {primera.isoformat()} | última fecha_hora: {ultima.isoformat()}")
else:
    print("Paso omitido (accion =", accion + ")")

## Paso 8 — Verificación

Se muestran el catálogo, el conteo de la bitácora y los archivos de la carpeta de entrada. Después de `reiniciar` o de una primera preparación, todas las `ultima_marca` están vacías y la bitácora en cero.

In [ ]:
print("== Catálogo ==")
display(spark.table(CAT).orderBy("orden", "id"))
print("== Filas en la bitácora:", spark.table(BIT).count())
print("== Archivos en", ENTRADA_TX)
for f in notebookutils.fs.ls(ENTRADA_TX):
    print(f"  {f.name}  ({f.size} bytes)")

## Paso 9 — Salida con un resumen en JSON

`notebookutils.notebook.exit` devuelve un resumen (acción, esquema, catálogo con sus marcas, filas de la bitácora y archivos de entrada). Cuando el cuaderno se lanza por la API (o desde un pipeline), ese texto queda disponible como `exitValue` y permite comprobar el resultado sin abrir el cuaderno. Va en su propia celda y fuera de cualquier `try/except`.

In [ ]:
resumen = {
    "accion": accion,
    "esquema": ESQUEMA or "ninguno",
    "catalogo": [
        {"tabla": r["tabla_destino"], "tipo": r["tipo_carga"], "activo": r["activo"], "ultima_marca": r["ultima_marca"]}
        for r in spark.table(CAT).orderBy("orden", "id").collect()
    ],
    "filas_bitacora": spark.table(BIT).count(),
    "archivos_entrada": [f.name for f in notebookutils.fs.ls(ENTRADA_TX)],
}
if accion == "incremento":
    resumen["incremento"] = {"archivo": destino_inc, "filas": cantidad, "primera": primera.isoformat(), "ultima": ultima.isoformat()}
notebookutils.notebook.exit(json.dumps(resumen, ensure_ascii=False))

## Siguiente paso

Ejecutar el pipeline `FAC_PL_Ingesta_Metadata` (o el cuaderno `FAC_NB_Carga_Generica` a mano con los valores de una fila del catálogo). Para el guion de la demo y los criterios de prueba, ver `README-metadata-driven.md`.

> Para mostrar el incremental: ejecuta el pipeline, vuelve a ejecutarlo (0 filas nuevas), cambia `accion` a `incremento`, ejecuta este cuaderno y vuelve a lanzar el pipeline (solo se carga lo nuevo). Para empezar de cero usa `accion = "reiniciar"`.